In [14]:
import sys

!{sys.executable} -m pip install "langchain>=1.2.0" langchain-community langchain-openai langchain-text-splitters pypdf chromadb tiktoken

In [15]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/mahsa/Documents/RAG - Cursor/RAG Project/.env")
api_key = os.getenv("OPENAI_API_KEY")
print("API key Loaded?", bool(api_key))

API key Loaded? True


In [16]:
PDF_PATH = "/Users/mahsa/Documents/RAG - Cursor/RAG Project/ki-strategi_en.pdf"

In [ ]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
import hashlib

# Load PDF
print("Loading and processing PDF...")
try : 
    #Load the PDF
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()
    print(f"Successfully loaded {len(docs)} pages")

    #Enhance metadata for each page:
    for i,doc in enumerate(docs):
        #add source file information
        doc.metadata["source"] = PDF_PATH

        #ensure page number is recorded
        doc.metadata["page"] = i

        #creat a content fingerprint to detect duplicates
        content_preview = doc.page_content[:100]
        doc.metadata["content_hash"] = hashlib.md5(
            content_preview.encode("utf-8")
        ).hexdigest()

    print("Enhanced metadata for all pages")
    print(f"sample - page 0 metadata: {docs[0].metadata}")

except FileNotFoundError:
    print(f"Error: pdf file not found at {PDF_PATH}")
    print("please check if the file path is correct")
    sys.exit(1)

except Exception as e :
    print(f"Unexpected error loading PDF: {e}")
    sys.exit(1)





Loading and processing PDF...


Successfully loaded 67 pages
Enhanced metadata for all pages
sample - page 0 metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-01-17T11:09:08+01:00', 'author': 'Christine Hafskjold', 'moddate': '2020-01-17T11:09:08+01:00', 'source': '/Users/mahsa/Documents/RAG - Cursor/RAG Project/ki-strategi_en.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1', 'content_hash': '97e0d64af3204ee9613dc32615d9e5db'}
Loaded 67 pages, split into 280 chunks.


In [22]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function = len,
    separators = [
        "\n\n", "\n", ".", "!", ",", " ", ""],
        keep_separator = True,
        strip_whitespace = True, )
split_docs = splitter.split_documents(docs)
print(f"Loaded {len(docs)} pages, split into {len(split_docs)} chunks.")

#analyze the chunks to undrestand our splitting
print("\n Chunk Analysis:")
print(f" - Average chunk length: {sum(len(doc.page_content) for doc in split_docs) / len(split_docs):.0f} characters")
print(f" - Shortest chunk: {min(len(doc.page_content) for doc in split_docs)} characters")
print(f" - Longest chunks : {max(len(doc.page_content) for doc in split_docs)} characters")

#show example of how chunks maintain context
print("\n Example of chunk boundries :")
example_doc = next(doc for doc in split_docs if len(doc.page_content) > 500)
print(f" Chunk from page {example_doc.metadata.get('page', '?')}:")
print(f"\"{example_doc.page_content[:200]}...\"")
print(f" [continues for {len(example_doc.page_content)} total characters]")

#verify metadata was preserved
print(f"\n Metadata preserved : page {split_docs[0].metadata.get('page')} still has source {split_docs[0].metadata.get('source')[:30]}...")


Loaded 67 pages, split into 235 chunks.

 Chunk Analysis:
 - Average chunk length: 861 characters
 - Shortest chunk: 111 characters
 - Longest chunks : 998 characters

 Example of chunk boundries :
 Chunk from page 1:
"2 
Foreword 
It is difficult to predict the future, but we know that Norway will be 
affected by the age wave, climate change and increasing global-
isation, and that in the coming years we must work ..."
 [continues for 954 total characters]

 Metadata preserved : page 0 still has source /Users/mahsa/Documents/RAG - C...


In [30]:
# Vector store with OpenAI embeddings

#define current folder to save the vector store
PERSIST_DIRECTORY = "./chroma_db"


#initialize embeddings
embeddings = OpenAIEmbeddings()

#check if we already have a saved vector store
if os.path.exists(PRESIST_DIRECTORY) :
    print(f" Found existing vectore store at '{PRESIST_DIRECTORY}'")
    print(" Loading from disk...")

    #load the existing database
    vectordb = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        embedding_function = embeddings
    )
    print(f" Loaded existing vector store with {vectordb._collection.count()} chunks")

else:
    print("No existing vector store found. creating new one")
    print("This will take a moment and use OpenAI API calls")

    #create new database from documents
    vectordb = Chroma.from_documents(
        documents = split_docs,
        embedding = embeddings,
        persist_directory=PERSIST_DIRECTORY
    )

    #save to dist for next time
    vectordb.persist()
    print(f"Created and presisted new vector store with {len(split_docs)} chunks")
    print(f" saved to {PRESIST_DIRECTORY} for future use")


#create retriever with MMR for better results
print("\n Setting up MMR retriever...")
retriever = vectordb.as_retriever(
    search_type = "mmr",
    search_kwargs = {
        "k" : 5,
        "fetch_k" : 20,
        "lambda_mult" : 0.5
    }
)

print("Retriever configured with MMR diversity")

#Test with a sample query
print("\n" + "="* 50)
print("Testing Retriever with sample query: 'What is this document about?'")
test_docs = retriever.invoke("what is this document about?")

print(f" Found {len(test_docs)} relevant chunks : ")
for i,doc in enumerate(test_docs,1) :
    source_page = doc.metadata.get('page', 'unknown')
    content_preview = doc.page_content[:100].replace('\n', ' ')+ "..."
    print(f" {i}. page {source_page}: {content_preview}")

print("\n Vector store setup complete!")






 Found existing vectore store at './chroma_db'
 Loading from disk...
 Loaded existing vector store with 235 chunks

 Setting up MMR retriever...
Retriever configured with MMR diversity

Testing Retriever with sample query: 'What is this document about?'


 Found 5 relevant chunks : 
 1. page 66: Published by :  Ministry of Local Government and Modernisation     Additional copies may  be ordered...
 2. page 17: Source: Norwegian Tax Administration   White paper on the data-driven economy  The Government will p...
 3. page 25: they are rule-based. The regulations are programmed into the solution, making it  possible to give r...
 4. page 56: house in order36, meaning that they gain an overview of what data they manage, what  the data means,...
 5. page 42: 43  3.2 Skills  The Government wants digital skills and technology literacy to be given more  promin...

 Vector store setup complete!


In [40]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

#create an enhanced prompt with system message
prompt = ChatPromptTemplate.from_messages([(
    "system", """you are a precise document analyst, Your task is to answer questions using ONLY the provided context.

    **Core Rules:**
    1.Stick to the contect: Never use outside knowledge or make up information
    2.Be honest: If the context does not contain the answer, say "I cannot find this information in the provided document"
    3.Cite sources: A'ways mention the page number(s) where you found the information
    4.Be concise about complete: Give thorough answers without unnecessary fluff

    **Response Format Guidelines:**
    - For lists or multiple points, use bullet points (•)
    - For definitions or explanations, use clear paragraphs
    - If quoting, use "quotation marks" and cite the page
    - Structure complex answers with clear sections

     **What to Avoid:**
    - Don't add information from outside the document
    - Don't guess or speculate
    - Don't use markdown formatting (like **bold** or # headings)
    - Don't be overly verbose

    Remember: Quality over quantity. A precise, accurate answer is better than a long, speculative one."""
), 
("human", "Here is the context from the document:\n\n{context}\n\nBased ONLY on this context, please answer: {question}")
])

print("created enhanced prompt with system message and guidlines")


def format_docs_with_sources(docs):
    """Format documents with clear page number indicators"""
    formatted_chunk = []
    for doc in docs:
        page_num = doc.metadata.get('page', 'unknown')
        source = doc.metadata.get('source', 'unknown').split('/')[-1]
        formatted_chunk.append(f"[From page {page_num} of {source}] \n {doc.page_content}")
    return "\n\n---\n\n".join(formatted_chunk)

print("created documnet formatter with source citations")

parser = StrOutputParser()

rag_chain = (
    {"context": retriever | format_docs_with_sources,
    "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

print("Enhanced RAG chain created")

#Test with different type of queries
print("-"*50)
print("\n Testing enhanced prompt with different query types:")

query1 = " give me a short summary of this PDF."
print(f"\n Query 1 : '{query1}'")
answer1 = rag_chain.invoke(query1)
print(f"Answer:\n{answer1}")

query2 = "what are the main topics covered in this document?"
print(f"\n Query 2 : '{query2}'")
answer2 = rag_chain.invoke(query2)
print(f"Answer:\n{answer2}")

query3 = "what os the author's favorite color?"
print(f"\n Query 3 : '{query3}'")
answer3 = rag_chain.invoke(query3)
print(f"Answer:\n{answer3}")



created enhanced prompt with system message and guidlines
created documnet formatter with source citations
Enhanced RAG chain created
--------------------------------------------------

 Testing enhanced prompt with different query types:

 Query 1 : ' give me a short summary of this PDF.'
Answer:
The document appears to be a report published by the Ministry of Local Government and Modernisation, focusing on artificial intelligence (AI) and its implications. It includes sections on the definition of AI, how it works, and the importance of data management. The report emphasizes the growing need for high-performance computing due to the increasing volume of data in research and public administration. Additionally, it discusses the necessity for new buildings to be equipped for high-speed networks and outlines plans for white papers on various topics, including language resources and electronic communication. The publication is dated January 2020 and includes references to cooperation wit